# GOAL.MD — Molecular Dynamics Module

The `goal.md` module provides production-grade molecular dynamics simulations
tightly integrated with the `goal.ml` machine-learning framework.

## Three usage patterns

| Pattern | When to use |
|---------|-------------|
| **Config-first** | Reproducible experiments, CLI, batch runs |
| **Direct import** | Notebooks, interactive exploration, custom workflows |
| **MTS (multi-timescale)** | Expensive QM + fast ML, adaptive online learning |

## Install MD dependencies
```bash
pip install 'goal[md]'
# or with pixi:
pixi install
```

---
## Pattern 1 — Direct import (factory pattern)

In [ ]:
from goal.md import (
    MoleculeFactory,
    CalculatorFactory,
    DynamicsFactory,
    Simulation,
)

print("Available calculators:", list(CalculatorFactory._builders.keys()))
print("Available dynamics:   ", list(DynamicsFactory._builders.keys()))

In [ ]:
# ── Create a molecule from SMILES ────────────────────────────────────────────
# Requires: rdkit
atoms = MoleculeFactory.create("from_smiles", smiles="CCO")   # ethanol
print(f"Ethanol: {len(atoms)} atoms, symbols: {atoms.get_chemical_symbols()}")

In [ ]:
# ── Create a molecule from an XYZ file ───────────────────────────────────────
from ase.build import molecule as ase_molecule
from ase.io import write as ase_write
import tempfile, pathlib

water = ase_molecule("H2O")
tmp = pathlib.Path(tempfile.mkdtemp()) / "water.xyz"
ase_write(str(tmp), water)

atoms_from_file = MoleculeFactory.create("from_file", path=str(tmp))
print(f"Water from file: {len(atoms_from_file)} atoms")

---
## Pattern 2 — Running a simulation

The `Simulation` class ties together molecule + calculator + dynamics.
Here we use ASE's built-in EMT calculator (no external deps) as a stand-in.

In [ ]:
from ase.build import bulk
from ase.calculators.emt import EMT
import tempfile, pathlib

outdir = pathlib.Path(tempfile.mkdtemp())

# Copper FCC bulk — works with EMT out of the box
cu_atoms = bulk("Cu", "fcc", a=3.6) * (2, 2, 2)

sim = Simulation(
    atoms=cu_atoms,
    calculator=EMT(),
    steps=200,
    temperature_K=300.0,
    timestep_fs=1.0,
    log_interval=20,
    trajectory_file="md.traj",
    log_file="md.log",
    output_dir=str(outdir),
    show_progress=True,
)

result = sim.run()
print(f"Steps completed : {result.steps_completed}")
print(f"Final energy    : {result.final_energy:.4f} eV")
print(f"Final temperature: {result.final_temperature:.1f} K")
print(f"Energy samples  : {len(result.energy_history)}")

In [ ]:
# ── Plot energy over time ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.plot(result.energy_history, lw=1.5)
plt.xlabel("Sample")
plt.ylabel("Potential energy (eV)")
plt.title("Cu bulk — NVT 300 K")
plt.tight_layout()
plt.show()

---
## Pattern 3 — Loading a goal.ml trained model

Once you have trained a model with `goal-train`, load it as an ASE calculator
via the `goal.md.adapters` bridge.

In [ ]:
# ── Load a goal.ml checkpoint ─────────────────────────────────────────────────
# Uncomment and set the real path after training.

# from goal.md.adapters.model_loader import load_goal_calculator, get_model_cutoff
#
# ckpt = "outputs/train/2024-01-15_EnergyModel/last.ckpt"
# calc = load_goal_calculator(ckpt, device="cuda")
#
# cutoff = get_model_cutoff(ckpt)
# print(f"Model cutoff: {cutoff} Å")
#
# sim = Simulation(
#     atoms=atoms,
#     calculator=calc,
#     steps=5000,
#     temperature_K=300.0,
# )
# result = sim.run()

print("Set ckpt path above to use a trained goal.ml model.")

---
## Pattern 4 — Data conversion: ASE ↔ goal.ml

`goal.ml` uses `AtomicGraph` (pure tensors) internally.
Use the adapters in `goal.md.adapters` to convert between formats.

In [ ]:
from goal.md.adapters import atomic_graph_from_ase, ase_atoms_from_atomic_graph
from ase.build import molecule as ase_mol

water = ase_mol("H2O")

# ASE Atoms → AtomicGraph (for ML training/analysis)
graph = atomic_graph_from_ase(water, cutoff=5.0)
print(f"AtomicGraph: {graph.pos.shape[0]} atoms, {graph.edge_index.shape[1]} edges")

# AtomicGraph → ASE Atoms (back to simulation-land)
water_back = ase_atoms_from_atomic_graph(graph)
print(f"Back to ASE: {len(water_back)} atoms")

---
## Pattern 5 — Multi-Timescale MD (RESPA)

MTS-MD uses two force levels:
- **Fast** (inner loop): cheap ML model — called every inner step
- **Slow** (outer loop): expensive QM correction = `F_QM - F_ML` — called every `mts_ratio` steps

The physics follows the RESPA (Reference System Propagator Algorithm) by Tuckerman et al. (1992),
which is the same equation set implemented in **i-pi**.

In [ ]:
from goal.md.mts.integrators.respa import RespaIntegrator
from ase.build import bulk
from ase.calculators.emt import EMT
from ase import units
import numpy as np

atoms = bulk("Cu", "fcc", a=3.6) * (2, 2, 2)

# In real use: fast_calc = GOALCalculator(...), slow_calc = ORCA(...)
# Here we demonstrate with EMT for both levels.
fast_calc = EMT()
slow_calc = EMT()

dyn = RespaIntegrator(
    atoms=atoms,
    fast_calculator=fast_calc,
    timestep=1.0 * units.fs,       # inner (fast) timestep
    mts_ratio=4,                    # slow step every 4 fast steps
    slow_calculator=slow_calc,
    temperature_K=300.0,
    friction=0.01,
)

# Run 20 outer steps (= 80 inner steps at 1 fs = 20 fs effective outer dt)
dyn.run(20)

print("MTS stats:", dyn.mts_stats)
print("Final energy:", atoms.get_potential_energy(), "eV")

---
## Pattern 6 — High-level MTSSimulation

The `MTSSimulation` class bundles everything including i-pi unit conversion,
observers, progress bars, and a config-first constructor.

In [ ]:
from goal.md.mts.ipi_adapter import MTSSimulation, MTSConfig
from ase.build import bulk
from ase.calculators.emt import EMT
import tempfile, pathlib

outdir = pathlib.Path(tempfile.mkdtemp())
atoms = bulk("Cu", "fcc", a=3.6) * (2, 2, 2)

cfg = MTSConfig(
    timestep_fs=1.0,
    mts_ratio=4,
    total_steps=40,            # outer steps (keep small for demo)
    temperature_K=300.0,
    log_interval=10,
    trajectory_file="mts.traj",
    output_dir=str(outdir),
    show_progress=True,
)

sim = MTSSimulation(
    atoms=atoms,
    fast_calculator=EMT(),   # replace with goal_model in real use
    slow_calculator=EMT(),   # replace with ORCA/CP2K in real use
    config=cfg,
)

result = sim.run()
print("Steps:", result["steps_completed"])
print("Final energy:", result["final_energy"], "eV")
print("MTS stats:", result["mts_stats"])

---
## Pattern 7 — Adaptive online-learning MTS

The `QMLearnerCalculator` trains an ML model on-the-fly from QM data.
The `ThresholdPolicy` switches from QM to ML once force RMSE drops below
the threshold.

In [ ]:
from goal.md.mts.dual_calculator.qm_ml_learner import (
    QMLearnerCalculator,
    ThresholdPolicy,
    AlwaysBasePolicy,
)
from ase.build import bulk
from ase.calculators.emt import EMT
from ase import units
from ase.md.langevin import Langevin

atoms = bulk("Cu", "fcc", a=3.6) * (2, 2, 2)

# policy: switch to ML when force RMSE < 0.1 eV/Å and energy MAE < 0.01 eV/atom
policy = ThresholdPolicy(force_rmse=0.1, energy_mae=0.01, min_steps=5)

learner = QMLearnerCalculator(
    base_calculator=EMT(),   # replace with ORCA for real QM
    ml_calculator=EMT(),     # replace with GOALCalculator for real ML
    policy=policy,
    eval_interval=5,
    data_buffer_size=100,
)

atoms.calc = learner
dyn = Langevin(atoms, timestep=1.0 * units.fs, temperature_K=300, friction=0.01)
dyn.run(20)

history = learner.get_training_history()
print("Buffer size :", history["data_buffer_size"])
print("Active calc :", history["active_calculator"])
print("Force RMSE  :", history["current_force_rmse"])
print("Energy MAE  :", history["current_energy_mae"])

---
## Config-first workflow summary

For reproducible experiments, use the Hydra configs in `configs/md/`:

```bash
# Standard Langevin MD with a trained goal.ml model
goal-simulate calculator.checkpoint=outputs/train/run/last.ckpt

# MTS with QM correction
goal-simulate-mts \
    fast_calculator.checkpoint=outputs/train/run/last.ckpt \
    slow_calculator.orca_path=/path/to/orca \
    mts.mts_ratio=4

# Online-learning adaptive MTS
goal-simulate-mts --config-name md/mts/simulations/online_learner \
    ml_calculator.checkpoint=outputs/train/run/last.ckpt \
    base_calculator.orca_path=/path/to/orca
```

### Key config files

| Config | Purpose |
|--------|---------|
| `configs/md/simulations/langevin_with_model.yaml` | Standard Langevin |
| `configs/md/mts/simulations/mts_respa.yaml` | RESPA MTS |
| `configs/md/mts/simulations/online_learner.yaml` | Adaptive QM/ML |
| `configs/md/calculators/*.yaml` | Calculator templates |
| `configs/md/molecules/*.yaml` | Molecule templates |